# Part 4 — AlphaEarth pilot (25 patches) and sub-pixel offsets

**Goal:**
1. download AlphaEarth 2019 (`GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL`) for the 25 patches of the RF pilot, with `ee.data.computePixels` on the same grid as TESSERA (UTM of the tile, top-left snapped to 10 m), and check it;
2. measure the sub-pixel offset between each embedding and the PASTIS parcels, per tile: TESSERA on all patches (2019, 2018) and AlphaEarth on the pilot.

**Needs:** a Google Cloud project registered for Earth Engine (noncommercial: earthengine.google.com/noncommercial), and Parts 1 and 3 on the Drive. **Runtime:** CPU. **Time:** about 10 min (the AlphaEarth speed is measured here).


Data attribution: "The AlphaEarth Foundations Satellite Embedding dataset is produced by Google and Google DeepMind." (CC-BY 4.0)

## 1. Runtime, Drive and configuration
Set `GEE_PROJECT` to the ID of your Earth Engine project.

In [ ]:
import sys, shutil, platform, runpy
from pathlib import Path
import json, numpy as np, pandas as pd
print("Python", platform.python_version())

from google.colab import drive
drive.mount("/content/drive")

GEE_PROJECT = "your-project-id"   # <-- EDIT
DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"
CODE = Path("/content/code")

PROC = Path("/content/processed")
PROC_DRIVE = DRIVE / "processed"
PILOT = DRIVE / "pilot"
RESULTS = DRIVE / "results"

## 2. Code

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt

def run_script(name, *args):
    # Runs scripts/<name>.py inside this kernel, so that it uses the Earth Engine login of step 4.
    sys.argv = [name] + [str(a) for a in args]
    runpy.run_path(str(CODE / "scripts" / f"{name}.py"), run_name="__main__")

## 3. Outputs of Parts 1 and 3

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)
for f in ["meta.csv", "instances.npy", "selection_seed42.json", "qa_tessera_2019.csv", "qa_tessera_2018.csv"]:
    shutil.copy(PROC_DRIVE / f, PROC / f)
meta = pd.read_csv(PROC / "meta.csv", index_col="idx")
PILOT_IDS = [p["patch_id"] for p in json.loads((PROC / "selection_seed42.json").read_text())["patches"]]
print(len(meta), "patches,", len(PILOT_IDS), "pilot patches")

## 4. Earth Engine login
A window asks you to allow access with your Google account.

**Check:** a positive number of AlphaEarth images is printed for 2019.

**Common errors:** `GEE_PROJECT` not edited; the project is not registered for Earth Engine, or the Earth Engine API is not enabled in it (the error message says which).

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
n = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL").filterDate("2019-01-01", "2020-01-01").size().getInfo()
print(n, "AlphaEarth images for 2019")

## 5. Grid check (no request sent)
The request grid of the first pilot patch: `translateX` / `translateY` must be `x0 - 10` / `y1 + 10` of that patch (one pixel of margin for the shift test), `crsCode` its UTM zone, size 130 × 130.

In [ ]:
run_script("fetch_alphaearth", "--processed", PROC, "--year", 2019,
           "--shard-dir", PILOT / "ae_2019", "--dry-run", "--ids", *PILOT_IDS)
print(meta.loc[meta.ID_PATCH.isin(PILOT_IDS), ["ID_PATCH", "TILE", "epsg", "x0", "y1"]].head(1).to_string(index=False))

## 6. AlphaEarth pilot
**Check:**
- the number of images and the properties of one image are printed (write down `DATASET_VERSION` if it appears);
- `errors: 0`;
- in the table: empty `error`, `nodata_frac` = 0, `norm_dev` close to 0 (AlphaEarth vectors have unit length), `ratio_+0_+0` in the same range as TESSERA.

In [ ]:
run_script("fetch_alphaearth", "--project", GEE_PROJECT, "--processed", PROC, "--year", 2019,
           "--shard-dir", PILOT / "ae_2019", "--ids", *PILOT_IDS)

In [ ]:
d = np.load(PILOT / "ae_2019" / "pilot_0000.npz")
qa = pd.DataFrame({k[3:]: d[k] for k in d.files if k.startswith("qa_")})
qa.insert(0, "ID_PATCH", meta.loc[d["idx"], "ID_PATCH"].values)
qa.insert(1, "TILE", meta.loc[d["idx"], "TILE"].values)
pd.set_option("display.width", 200)
print(qa[["ID_PATCH", "TILE", "error", "nodata_frac", "norm_dev", "best_dx", "best_dy", "ratio_+0_+0", "seconds"]]
      .to_string(index=False))
print(f"\nbest shift (0, 0): {((qa.best_dx == 0) & (qa.best_dy == 0)).sum()}/{len(qa)}")
print(f"estimated time for 2,433 patches with 8 workers: {qa.seconds.median() * 2433 / 8 / 60:.0f} min")
emb = d["emb"]
print("dtype", emb.dtype, "| value range", np.nanmin(emb), np.nanmax(emb))

## 7. Sub-pixel offsets
`subpixel_offset.py` fits a parabola through the ratios at shifts −1, 0, +1 of each patch and takes the median per tile; `pred_dx` / `pred_dy` are the offsets expected if the PASTIS grid starts at the raw footprint corner. `res_snapped` and `res_raw` are the remaining misregistration (pixels) with the snapped origin (current data) and with the raw origin.

**Check:** the TESSERA rows reproduce the table (e.g. 2019 t31tfj: `med_dy` −0.293), and the AlphaEarth rows are printed. With about 6 pilot patches per tile, the AlphaEarth rows are only indicative; the full AlphaEarth run will give the real values.

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
!python scripts/subpixel_offset.py --processed "{PROC}" \
    --qa "{PROC}/qa_tessera_2019.csv" "{PROC}/qa_tessera_2018.csv" "{PILOT}/ae_2019/pilot_0000.npz" \
    --out "{RESULTS}/subpixel_offset.csv"